[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/57_on_policy_distillation_loss_solution.ipynb)

# Solution: On-Policy Distillation Loss

Reference solution.


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F


In [ ]:
# ✅ SOLUTION

def on_policy_distillation_loss(student_logits, teacher_logits, completion_mask,
                                temperature=2.0):
    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_probs = F.softmax(teacher_logits.detach() / temperature, dim=-1)
    token_kl = (teacher_probs * (torch.log(teacher_probs.clamp_min(1e-12)) - student_log_probs)).sum(dim=-1)
    token_kl = token_kl * (temperature ** 2)
    return (token_kl * completion_mask).sum() / completion_mask.sum().clamp_min(1.0)


In [ ]:
student = torch.randn(2, 4, 8)
teacher = torch.randn(2, 4, 8)
mask = torch.tensor([[1, 1, 1, 1], [1, 1, 0, 0]], dtype=torch.float32)
print('Loss:', on_policy_distillation_loss(student, teacher, mask))


In [ ]:
from torch_judge import check
check('on_policy_distillation_loss')
